# SkillCompass — EDA

IT-аналитики (BA / SA / PA / DA): зарплаты, роли, источники HH + SuperJob.

Данные: `data/processed/vacancies.parquet`

**Перед запуском ноутбука** (из корня `jobpulse/`, Docker с PostgreSQL):

```powershell
python scripts/export_parquet_from_db.py
```

Альтернатива без БД: `python scripts/fetch_all.py` (прямой сбор в parquet).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
elif (ROOT / "ml").exists():
    pass
else:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from load_data import load_vacancies, prepare_salary_target

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

In [ ]:
df = load_vacancies()
df = prepare_salary_target(df)

if "role_label" not in df.columns and "analyst_role" in df.columns:
    labels = {
        "business_analyst": "Бизнес-аналитик",
        "systems_analyst": "Системный аналитик",
        "product_analyst": "Продуктовый аналитик",
        "data_analyst": "Аналитик данных",
    }
    df["role_label"] = df["analyst_role"].map(labels).fillna(df["analyst_role"])

with_salary = int(df["salary_mid"].notna().sum())
with_skills = int(df["key_skills"].notna().sum()) if "key_skills" in df.columns else 0

print(f"Вакансий: {len(df)}")
print(f"С зарплатой (salary_mid): {with_salary}")
print(f"С key_skills: {with_skills}")
print("Источники:", df["source"].value_counts().to_dict())
print("\nРоли:")
print(df["role_label"].value_counts() if "role_label" in df else df["analyst_role"].value_counts())
df.head()


## Гипотеза 1: зарплаты системных аналитиков выше бизнес-аналитиков

In [ ]:
paid = df.dropna(subset=["salary_mid"])
order = paid.groupby("role_label")["salary_mid"].median().sort_values(ascending=False).index
sns.boxplot(data=paid, x="role_label", y="salary_mid", order=order)
plt.xticks(rotation=15)
plt.title("Медианная зарплата по роли IT-аналитика")
plt.ylabel("₽ / мес")
plt.tight_layout()
plt.show()

## Гипотеза 2: доля вакансий с указанной зарплатой различается по источникам

In [ ]:
df["has_salary"] = df["salary_mid"].notna()
rate = df.groupby("source")["has_salary"].mean().mul(100).round(1)
rate.plot(kind="bar", color=["#6366f1", "#10b981"])
plt.title("% вакансий с указанной зарплатой")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
rate

## Гипотеза 3: опыт влияет на зарплату

In [ ]:
exp = paid.dropna(subset=["experience"])
sns.barplot(data=exp, x="experience", y="salary_mid", estimator="median", errorbar=None)
plt.xticks(rotation=20)
plt.title("Медианная зарплата по опыту")
plt.tight_layout()
plt.show()

## Выводы

1. Рынок IT-аналитиков неоднороден: разные роли дают разные медианы.
2. HH и SuperJob по-разному указывают зарплату в объявлениях.
3. Для ML используем `analyst_role`, `experience`, `area`, `source`, `salary_spec`.
4. Объём данных ~3000+ вакансий после полного ingest и enrich (навыки HH).
